# Agent 第 04 课：多工具、工具路由、工具设计心法

上一课的 agent 只有 3 个工具，模型基本不会选错。真实场景下你可能有 **20 个、50 个、甚至动态加载**的工具，这时候各种病就来了：

- **选错工具**：有 `search_web` 和 `search_internal_wiki`，模型乱选
- **参数错**：name vs id 搞反、日期格式错、必填字段漏了
- **重复调用**：同一个工具连调 5 次结果一样
- **不调用**：明明有工具非要靠直觉答

这一课不加新算法，而是讲**怎么设计工具集**让模型用对。工具设计是 agent 工程里被严重低估的一环——**80% 的 agent bug 其实是工具 schema 没写好**。

## 1. 工具设计五条黄金法则

这几条是我和社区经验的浓缩，每一条都有血泪：

### 法则 1：**工具名 = 动词短语，描述具体动作**
- ❌ `user`、`db`、`info`（名词，含糊）
- ✅ `get_user_by_id`、`query_orders_in_range`、`send_slack_message`

### 法则 2：**description 写"什么时候用"，不是"这个工具是什么"**
- ❌ `'A function that searches the web.'`
- ✅ `'Use this when the user asks about current events, recent news, or facts after 2024. Not for static facts like geography or math.'`

### 法则 3：**参数越少越好；必填的写 required，可选的给默认值**
模型多一个参数就多一次出错的机会。能合并就合并。

### 法则 4：**返回结构化 JSON，而不是自由文本**
- ❌ `"Found 3 orders: #1001 ($50, paid), #1002 ($20, pending), ..."`
- ✅ `[{'id':1001,'amount':50,'status':'paid'},...]`
结构化返回模型下一步容易正确解析和引用。

### 法则 5：**失败时返回明确的 error 而不是 raise**
工具函数不能向 agent loop 抛异常。要返回 `{'error': '...'}`，让模型看到并决定重试/换策略。

In [ ]:
import json, re
from openai import OpenAI
from pydantic import BaseModel, ValidationError

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 2. 一个假公司数据集 + 工具集

我们模拟一个小电商，有用户、订单、产品三张表。给 agent 设计 4 个工具。

In [ ]:
USERS = {
    1: {'id':1,'name':'Alice','country':'US'},
    2: {'id':2,'name':'Bob','country':'UK'},
    3: {'id':3,'name':'Carol','country':'US'},
}
ORDERS = [
    {'order_id':1001,'user_id':1,'product_id':'SKU-A','amount':50.0,'status':'paid'},
    {'order_id':1002,'user_id':1,'product_id':'SKU-B','amount':20.0,'status':'pending'},
    {'order_id':1003,'user_id':2,'product_id':'SKU-A','amount':50.0,'status':'paid'},
    {'order_id':1004,'user_id':3,'product_id':'SKU-C','amount':120.0,'status':'refunded'},
]
PRODUCTS = {
    'SKU-A': {'sku':'SKU-A','name':'Notebook','price':50.0},
    'SKU-B': {'sku':'SKU-B','name':'Pen','price':20.0},
    'SKU-C': {'sku':'SKU-C','name':'Backpack','price':120.0},
}

## 3. 工具实现（带参数校验）

用 pydantic 做参数校验。失败时不 raise，而是**把错误返回给 agent**。

In [ ]:
class GetUserArgs(BaseModel):
    user_id: int

class GetOrdersArgs(BaseModel):
    user_id: int
    status: str | None = None  # optional filter

class GetProductArgs(BaseModel):
    sku: str

class SearchUsersArgs(BaseModel):
    name_contains: str

def tool_get_user(a): return USERS.get(a.user_id, {'error':f'no user {a.user_id}'})
def tool_get_orders(a):
    rows = [o for o in ORDERS if o['user_id']==a.user_id]
    if a.status: rows = [o for o in rows if o['status']==a.status]
    return rows
def tool_get_product(a): return PRODUCTS.get(a.sku, {'error':f'no sku {a.sku}'})
def tool_search_users(a):
    q = a.name_contains.lower()
    return [u for u in USERS.values() if q in u['name'].lower()]

TOOLS = [
    {'name':'get_user',
     'description':'Fetch a single user by numeric user_id. Use when you already know the user_id. For looking up by name, use search_users first.',
     'parameters':GetUserArgs.model_json_schema()},
    {'name':'search_users',
     'description':'Find users whose name contains a substring (case-insensitive). Use this when the user is referred to by name, not id.',
     'parameters':SearchUsersArgs.model_json_schema()},
    {'name':'get_orders',
     'description':'List all orders for a given user_id. Optional status filter: paid, pending, refunded.',
     'parameters':GetOrdersArgs.model_json_schema()},
    {'name':'get_product',
     'description':'Fetch product info by SKU (e.g. SKU-A).',
     'parameters':GetProductArgs.model_json_schema()},
]

TOOL_IMPL = {
    'get_user': (GetUserArgs, tool_get_user),
    'search_users': (SearchUsersArgs, tool_search_users),
    'get_orders': (GetOrdersArgs, tool_get_orders),
    'get_product': (GetProductArgs, tool_get_product),
}

## 4. Agent loop（从第 03 课拷过来，微调）

In [ ]:
SYSTEM = f"""You are a data lookup assistant. You have tools to query a small e-commerce database.

Tools:
{json.dumps(TOOLS, indent=2)}

Protocol:
- One tool call at a time, wrapped in <tool_call>...</tool_call>, then wait.
- If a tool returns an error, try a different approach.
- When done, reply with 'Final Answer: ...'.
- Do not invent data."""

TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)
FINAL_RE = re.compile(r'Final Answer:\s*(.+)', re.S)

def exec_tool(name, args_dict):
    if name not in TOOL_IMPL:
        return {'error': f'unknown tool {name}'}
    schema, fn = TOOL_IMPL[name]
    try:
        args = schema(**args_dict)
    except ValidationError as e:
        return {'error': f'invalid arguments: {e.errors()}'}
    try:
        return {'result': fn(args)}
    except Exception as e:
        return {'error': f'{type(e).__name__}: {e}'}

def run_agent(question, max_steps=10, verbose=True):
    messages = [{'role':'system','content':SYSTEM},{'role':'user','content':question}]
    for step in range(1, max_steps+1):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=messages)
        text = r.choices[0].message.content
        messages.append({'role':'assistant','content':text})
        m_tool = TOOL_CALL_RE.search(text)
        m_final = FINAL_RE.search(text)
        if verbose:
            print(f'--- step {step} ---'); print(text); print()
        if m_final:
            return m_final.group(1).strip()
        if m_tool:
            try: call = json.loads(m_tool.group(1))
            except json.JSONDecodeError as e:
                messages.append({'role':'user','content':f'Your tool_call JSON was invalid: {e}. Retry.'}); continue
            obs = exec_tool(call.get('name'), call.get('arguments') or {})
            if verbose: print('>>> tool result:', obs, '\n')
            messages.append({'role':'user','content':f'<tool_response>\n{json.dumps({"name":call.get("name"),"content":obs})}\n</tool_response>'})
            continue
        messages.append({'role':'user','content':'Issue a <tool_call> or give Final Answer: ...'})
    return None

## 5. 测试一：按名字查询（强制工具链）

问题："Bob 总共下了几单？都是什么状态？"

预期 agent 行为：先 `search_users('Bob')` 拿 id，再 `get_orders(user_id=2)`。

In [ ]:
print('ANSWER:', run_agent('How many orders has Bob placed, and what are their statuses?'))

## 6. 测试二：故意的模糊问题

问题："Alice 和 Carol 谁花得多？" — 需要两次 search + 两次 get_orders + 求和比较。

In [ ]:
print('ANSWER:', run_agent('Who spent more money in total on paid orders: Alice or Carol?'))

## 7. 测试三：不存在的用户（错误处理）

"告诉我 user_id 999 的所有订单。" — 工具会返回空列表，agent 应该老老实实说"没找到"，而不是编造。

In [ ]:
print('ANSWER:', run_agent('List all orders for user_id 999.'))

## 8. 工程直觉

1. **工具 description 是你唯一的"路由器"**。LLM 只靠 description 选工具。写得像文档注释一样认真（包括"什么时候**不**该用"）。
2. **工具数量有上限**。超过 20 个工具，即使顶级模型选择正确率也会下降。解决办法：**两层工具集**——一个 meta 工具 `list_available_tools(category)`，让 agent 先问"我该用哪一组"再调用具体工具。
3. **pydantic 校验 = agent 的 type system**。JSON schema 里声明了类型，但是模型偶尔还是会传 `"1"` 而不是 `1`——pydantic 会告诉你，然后你把错误扔回 agent 重试。
4. **工具返回的 JSON 字段名要稳定**。今天叫 `user_id`，明天改成 `uid`，prompt 里的例子对不上，模型就懵了。像对外 API 一样做版本管理。
5. **幂等工具 + 非幂等工具要区别对待**。`get_*` 重复调用无所谓；`send_email`、`delete_*` 要加"确认"步骤（第 10 课安全课会讲）。

---

下一课：**第 05 课 Agent 记忆**——当对话很长、或者 agent 要跨会话记住事情时，简单塞进 messages 就爆上下文了。讲短期记忆（滑动窗口、摘要压缩）和长期记忆（向量库 + 回调 RAG 课）。